In [ ]:
import pandas as pd
sales = pd.read_csv("sales_transactions.csv", parse_dates=["date"])
promos = pd.read_csv("promotion_calendar.csv", parse_dates=["start_date", "end_date"])
sales.info()

In [ ]:
promos

In [ ]:
print(sales["date"].min(), "to", sales["date"].max())
print(sales["category"].value_counts())
print(sales["promotion_id"].value_counts(dropna=False))
sales["discount_rate"].describe()

In [ ]:
odd = sales[sales["discount_rate"] > 1].merge(promos, on="promotion_id", how="left")
gross = odd["quantity"] * odd["unit_price"]
odd["as_fraction"] = gross * (1 - odd["discount_rate"])
odd["as_percent"] = gross * (1 - odd["discount_rate"] / 100)
odd[["transaction_id", "discount_rate", "planned_discount", "net_revenue", "as_fraction", "as_percent"]].round(2)

In [ ]:
df = sales.copy()
df["disc"] = df["discount_rate"].where(df["discount_rate"] <= 1, df["discount_rate"] / 100)
df = df.eval("""
gross = quantity * unit_price
calc_net = gross * (1 - disc)
calc_margin = calc_net - quantity * unit_cost
margin_pct = calc_margin / calc_net
diff = net_revenue - calc_net
""")
mismatch = df[df["diff"].abs() > 0.01]
mismatch[["transaction_id", "disc", "gross", "net_revenue", "calc_net", "gross_margin", "calc_margin"]].round(2)

In [ ]:
df["month"] = df["date"].dt.month
df.pivot_table(index="category", columns="month", values="disc", aggfunc="mean").round(2)

In [ ]:
g = df.groupby(["category", "month"])["disc"]
q1, q3, n = g.transform("quantile", 0.25), g.transform("quantile", 0.75), g.transform("size")
df["iqr_limit"] = q3 + 1.5 * (q3 - q1)
df["iqr_flag"] = (n >= 20) & (df["disc"] > df["iqr_limit"])
print("Groups below 20:", (g.size() < 20).sum(), "| rows not screened:", (n < 20).sum())
df[df["iqr_flag"]].groupby(["category", "month"]).size()

In [ ]:
v = df[df["promotion_id"].notna()].merge(promos, on="promotion_id", how="left")
v["id_ok"] = v["planned_discount"].notna()
v["category_ok"] = v["category"] == v["applicable_category"]
v["dates_ok"] = v["date"].between(v["start_date"], v["end_date"])
v["max_ok"] = v["disc"] <= v["max_discount"]
v["tolerance_ok"] = (v["disc"] - v["planned_discount"]).abs().round(4) <= 0.05
checks = ["id_ok", "category_ok", "dates_ok", "max_ok", "tolerance_ok"]
issues = v[~v[checks].all(axis=1)]
issues[["transaction_id", "category", "promotion_id", "disc", "iqr_flag"] + checks]

In [ ]:
ok = v["id_ok"] & v["category_ok"] & v["dates_ok"]
df["authorized"] = df["transaction_id"].isin(v.loc[ok, "transaction_id"])
df["over_30"] = (df["disc"] > 0.30) & ~df["authorized"]
df.loc[df["over_30"], ["transaction_id", "category", "month", "promotion_id", "disc", "iqr_limit", "iqr_flag"]].round(2)

In [ ]:
df["low_margin"] = df["margin_pct"] < 0.15
pd.crosstab(df.loc[df["low_margin"], "category"], df["authorized"], margins=True)

In [ ]:
df["promo"] = df["promotion_id"].where(df["authorized"], "None")
perf = df.groupby(["category", "promo"]).agg(sales=("quantity", "size"), avg_units=("quantity", "mean"),
                                             revenue=("calc_net", "sum"), profit=("calc_margin", "sum"))
perf["margin_pct"] = perf["profit"] / perf["revenue"]
perf.round(2)

In [ ]:
print(df[["net_revenue", "calc_net", "diff"]].sum().round(2))
print("Mismatches:", mismatch["diff"].sum().round(2))
print("Sum of categories:", df.groupby("category")["calc_net"].sum().sum().round(2))

In [ ]:
df["promo_issue"] = df["transaction_id"].isin(issues["transaction_id"])
df["data_issue"] = (df["diff"].abs() > 0.01) | (df["discount_rate"] > 1)
flags = ["iqr_flag", "data_issue", "promo_issue", "over_30", "low_margin"]
flagged = df[df[flags].any(axis=1)]
flagged.to_csv("flagged_transactions.csv", index=False)
flagged[flags].sum()

## Summary of findings
- **Data quality:** 8 discounts are stored as percentages (net revenue confirms the fraction), and 6 transactions record net revenue without the discount (+$58.35). Those source values were kept and are documented, not corrected.
- **Statistical anomalies:** 15 IQR outliers. 11 Toys groups have fewer than 20 transactions and were not screened. 3 outliers belong to an authorized Black Friday promotion.
- **Policy violations:** 17 promotion check failures (unknown PRM-99, PRM-07 on the wrong category, PRM-02 after its end date, PRM-05 above its maximum) and 10 discounts above 30% outside authorized promotions. IQR missed 3 of the 10.
- **Margin:** 72 transactions are below the 15% floor, 35 of them in authorized promotions. PRM-07 doubled the units per sale (3.3 vs 1.7) but kept only a 3% margin.
- **Recommendations:** review the flagged transactions, validate promotion IDs, categories, dates, and limits at checkout, require approval above 30%, and review the PRM-07 discount rate.

## Final notes
- The two synthetic input files were generated by Gemini before recording. The original files were not modified.